In [1]:
import pandas as pd
import numpy as np

## Задание 1

In [2]:
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_items = pd.read_csv('olist_order_items_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')
df_customers = pd.read_csv('olist_customers_dataset.csv')
df_translation = pd.read_csv('product_category_name_translation.csv')

df_orders['order_purchase_timestamp'] = pd.to_datetime(
    df_orders['order_purchase_timestamp'], errors = 'coerce'
)

**Стратегия для категорий без перевода**

таблицу переводов присоединяем через left join, поэтому ни один товар не теряется. Пропуск в английском названии возникает в двух случаях: у товара вообще не указана португальская категория, либо категория есть, но ее нет в справочнике переводов. Удалять такие товары через dropna нельзя, тк вместе с ними пропала бы реальная выручка и заказы клиентов, поэтому помечаем их отдельной категорией unknown, и тогда деньги остаются в расчетах, а сама категория видна в аналитике как отдельная группа

In [3]:
df_products = df_products.merge(
    df_translation, on = 'product_category_name', how = 'left'
)

no_category = df_products['product_category_name'].isna().sum()
no_translation = (
    df_products['product_category_name'].notna()
    & df_products['product_category_name_english'].isna()
).sum()

df_products['product_category_name_english'] = (
    df_products['product_category_name_english'].fillna('unknown')
)

unknown_count = (df_products['product_category_name_english'] == 'unknown').sum()
print(f'без категории: {no_category}, без перевода: {no_translation}')
print(f'товаров с категорией unknown: {unknown_count}')

без категории: 610, без перевода: 13
товаров с категорией unknown: 623


In [4]:
full = (
    df_items
    .merge(df_products, on = 'product_id', how = 'left')
    .merge(df_orders, on = 'order_id', how = 'left')
    .merge(df_customers, on = 'customer_id', how = 'left')
)

print(f'строк в order_items: {len(df_items)}')
print(f'строк в витрине: {len(full)}')
print(f'уникальных заказов в витрине: {full["order_id"].nunique()}')
full.head()

строк в order_items: 112650
строк в витрине: 112650
уникальных заказов в витрине: 98666


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,...,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,...,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,...,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,...,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,...,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,...,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP


**Гранулярность данных**

витрина full собрана на уровне позиции товара (order_id + order_item_id): одна строка равна одной единице товара в заказе. Если в заказе 3 товара, то все поля заказа (статус, дата, клиент) повторяются в трех строках, поэтому любые метрики уровня заказа нельзя просто суммировать по этой таблице. Сначала позиции агрегируются до уровня заказа, и только потом заказ присоединяется к клиенту

In [5]:
order_totals = df_items.groupby('order_id').agg(
    items_total = ('price', 'sum'),
    freight_total = ('freight_value', 'sum'),
    items_count = ('order_item_id', 'count')
).reset_index()

order_totals['total_order_value'] = (
    order_totals['items_total'] + order_totals['freight_total']
)
order_totals.head()

,order_id,items_total,freight_total,items_count,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,218.04


In [6]:
naive_revenue = (
    full
    .merge(order_totals[['order_id', 'total_order_value']], on = 'order_id', how = 'left')
    .query("order_status == 'delivered'")
    ['total_order_value']
    .sum()
)

**Бизнес-фильтр: только доставленные заказы**

Выручка это деньги, которые маркетплейс реально получил за выполненную продажу. отмененные и недоступные заказы (canceled, unavailable) до клиента не дошли, деньги по ним возвращаются или вообще не списываются, поэтому их стоимость завысит выручку. Заказы в статусах shipped, processing, invoiced еще в пути и тоже могут быть отменены или возвращены, то есть их сумма пока не является заработанной. кроме того, у отмененных заказов часто нет позиций в order_items, и после left join у них получается пустая стоимость, поэтому для финансового анализа оставляем только delivered

In [7]:
core = (
    df_orders
    .merge(order_totals, on = 'order_id', how = 'left')
    .merge(df_customers, on = 'customer_id', how = 'left')
)

print(core['order_status'].value_counts())
print(f'заказов без позиций: {core["total_order_value"].isna().sum()}')

core_delivered = core.loc[core['order_status'] == 'delivered'].copy()

print(f'всего заказов: {len(core)}')
print(f'доставлено: {len(core_delivered)}')
print(f'пропусков в стоимости у доставленных: {core_delivered["total_order_value"].isna().sum()}')
print(f'выручка: {core_delivered["total_order_value"].sum():,.2f} BRL')

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
заказов без позиций: 775
всего заказов: 99441
доставлено: 96478
пропусков в стоимости у доставленных: 0
выручка: 15,419,773.75 BRL


In [8]:
correct_revenue = core_delivered['total_order_value'].sum()

print(f'корректный расчёт: {correct_revenue:,.2f} BRL')
print(f'наивный расчёт: {naive_revenue:,.2f} BRL')
print(f'задвоение: {naive_revenue - correct_revenue:,.2f} BRL')

корректный расчёт: 15,419,773.75 BRL
наивный расчёт: 19,772,622.00 BRL
задвоение: 4,352,848.25 BRL


наивный расчет берет итоговую стоимость заказа и суммирует ее по витрине уровня позиций. заказ из 3 товаров попадает в сумму 3 раза вместе со всей доставкой, отсюда и завышение. корректный расчет суммирует total_order_value по таблице core, где каждый заказ встречается ровно один раз

## Задание 2

In [9]:
snapshot_date = core_delivered['order_purchase_timestamp'].max() + pd.Timedelta(days = 1)
print(f'дата среза: {snapshot_date}')

rfm = core_delivered.groupby('customer_unique_id').agg(
    recency = ('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
    frequency = ('order_id', 'nunique'),
    monetary = ('total_order_value', 'sum')
).reset_index()

rfm.describe().round(2)

дата среза: 2018-08-30 15:00:37


,recency,frequency,monetary
count,93358.00,93358.00,93358.00
mean,237.94,1.03,165.17
std,152.59,0.21,226.29
min,1.00,1.00,9.59
25%,114.00,1.00,63.01
50%,219.00,1.00,107.78
75%,346.00,1.00,182.51
max,714.00,15.00,13664.08


In [10]:
rfm['frequency'].value_counts().sort_index()

frequency
1     90557
2      2573
3       181
4        28
5         9
6         5
7         3
9         1
15        1
Name: count, dtype: int64

для Frequency квантили не подходят, тк у подавляющего большинства клиентов ровно 1 заказ, поэтому границы qcut совпадут и разбиение сломается. используем pd.cut с ручными границами: 1 заказ, 2 заказа, 3 заказа, 4 и более

In [11]:
rfm['R'] = pd.qcut(rfm['recency'], 4, labels = [4, 3, 2, 1]).astype(int)
rfm['F'] = pd.cut(
    rfm['frequency'], bins = [0, 1, 2, 3, np.inf], labels = [1, 2, 3, 4]
).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'], 4, labels = [1, 2, 3, 4]).astype(int)

rfm['RFM_score'] = (
    rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
)
rfm.head()

,customer_unique_id,recency,frequency,monetary,R,F,M,RFM_score
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,1,3,413
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,3,1,1,311
2,0000f46a3911fa3c0805444483337064,537,1,86.22,1,1,2,112
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1,211
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,1,4,214


In [12]:
customer_states = (
    df_customers[['customer_unique_id', 'customer_state']]
    .drop_duplicates(subset = 'customer_unique_id', keep = 'first')
)

champions = (
    rfm.loc[rfm['RFM_score'] == '444']
    .merge(customer_states, on = 'customer_unique_id', how = 'left')
)

print(f'чемпионов: {len(champions)} из {len(rfm)} ({len(champions) / len(rfm):.2%})')
print(champions['customer_state'].value_counts().head(5))

чемпионов: 22 из 93358 (0.02%)
customer_state
SP    8
RJ    4
RS    4
PE    2
PB    1
Name: count, dtype: int64


## Задание 3

In [13]:
items_delivered = full.loc[full['order_status'] == 'delivered']

top10 = (
    items_delivered
    .groupby('product_category_name_english')['price']
    .sum()
    .nlargest(10)
    .index
)

for i, cat in enumerate(top10, 1):
  print(f'{i}. {cat}')

1. health_beauty
2. watches_gifts
3. bed_bath_table
4. sports_leisure
5. computers_accessories
6. furniture_decor
7. housewares
8. cool_stuff
9. auto
10. toys


In [14]:
pivot = (
    items_delivered.loc[items_delivered['product_category_name_english'].isin(top10)]
    .pivot_table(
        index = 'customer_state',
        columns = 'product_category_name_english',
        values = 'price',
        aggfunc = 'sum',
        fill_value = 0
    )
)

pivot_share = pivot.div(pivot.sum(axis = 1), axis = 0) * 100

print(f'сумма по строке: {pivot_share.sum(axis = 1).round(2).unique()}')
pivot_share.round(2)

сумма по строке: [100.]


product_category_name_english,auto,bed_bath_table,computers_accessories,cool_stuff,furniture_decor,health_beauty,housewares,sports_leisure,toys,watches_gifts
customer_state,,,,,,,,,,
AC,6.03,6.33,13.81,1.12,14.68,15.47,5.72,18.71,2.62,15.50
AL,9.20,3.88,15.29,4.83,7.21,25.49,1.21,7.43,2.12,23.33
AM,6.52,6.14,16.36,5.76,1.94,25.01,3.68,12.28,6.50,15.82
AP,13.31,7.32,22.42,3.43,2.96,15.10,5.37,8.89,3.26,17.94
BA,9.92,8.50,11.12,6.86,7.88,17.02,5.65,12.61,4.89,15.55
CE,8.66,4.99,8.47,8.80,6.24,23.13,5.10,7.60,5.72,21.29
DF,9.09,8.51,12.56,6.37,6.33,15.56,6.76,11.83,5.97,17.01
ES,7.27,13.87,8.86,6.70,7.93,12.31,7.07,12.34,6.42,17.24
GO,6.35,12.51,7.76,9.82,6.75,15.70,8.27,10.79,4.38,17.67


In [15]:
country_avg = pivot_share['health_beauty'].mean()
country_std = pivot_share['health_beauty'].std()
deviation = (pivot_share['health_beauty'] - country_avg).sort_values(ascending = False)
z_score = deviation / country_std
top_state = deviation.index[0]

print(f'среднее по штатам: {country_avg:.2f}%')
print(f'аномалия: {top_state} — {pivot_share.loc[top_state, "health_beauty"]:.2f}% '
      f'(+{deviation.iloc[0]:.2f} п.п., z = {z_score.iloc[0]:.2f})')
print(pd.DataFrame({'deviation': deviation, 'z_score': z_score}).head(5).round(2))

среднее по штатам: 18.27%
аномалия: RN — 28.09% (+9.83 п.п., z = 1.88)
                deviation  z_score
customer_state                    
RN                   9.83     1.88
RO                   7.48     1.43
AL                   7.23     1.39
PE                   7.21     1.38
AM                   6.74     1.29


## Задание 4

In [16]:
cohorts = core_delivered.copy()

cohorts['order_month'] = cohorts['order_purchase_timestamp'].dt.to_period('M')
cohorts['cohort_month'] = (
    cohorts.groupby('customer_unique_id')['order_month'].transform('min')
)
cohorts['month_index'] = (
    (cohorts['order_month'].dt.year - cohorts['cohort_month'].dt.year) * 12
    + (cohorts['order_month'].dt.month - cohorts['cohort_month'].dt.month)
)

cohorts[['customer_unique_id', 'order_month', 'cohort_month', 'month_index']].head()

,customer_unique_id,order_month,cohort_month,month_index
0,7c396fd4830fd04220f754e42b4e5bff,2017-10,2017-09,1
1,af07308b275d755c9edb36a90c618231,2018-07,2018-07,0
2,3a653a41f6f9fc3d2a113cf8398680e8,2018-08,2018-08,0
3,7c142cf63193a1473d2e66489a9ae977,2017-11,2017-11,0
4,72632f0f9dd73dfee390c9b22eb56dd6,2018-02,2018-02,0


In [17]:
cohort_matrix = (
    cohorts
    .groupby(['cohort_month', 'month_index'])['customer_unique_id']
    .nunique()
    .reset_index()
    .pivot_table(
        index = 'cohort_month',
        columns = 'month_index',
        values = 'customer_unique_id'
    )
)

cohort_matrix

month_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2016-09,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,262.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.0,NaN,1.0,NaN,1.0,NaN,1.0,NaN,1.0,2.0,2.0
2016-12,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01,717.0,2.0,2.0,1.0,3.0,1.0,3.0,1.0,1.0,NaN,3.0,1.0,5.0,3.0,1.0,1.0,2.0,3.0,1.0,NaN
2017-02,1628.0,3.0,5.0,2.0,7.0,2.0,4.0,3.0,2.0,3.0,2.0,5.0,2.0,3.0,2.0,1.0,1.0,3.0,NaN,NaN
2017-03,2503.0,11.0,9.0,10.0,9.0,4.0,4.0,8.0,8.0,2.0,9.0,3.0,5.0,3.0,4.0,6.0,2.0,3.0,NaN,NaN
2017-04,2256.0,14.0,5.0,4.0,6.0,6.0,8.0,7.0,7.0,4.0,6.0,2.0,1.0,1.0,2.0,2.0,3.0,NaN,NaN,NaN
2017-05,3451.0,16.0,16.0,10.0,10.0,11.0,14.0,5.0,9.0,9.0,9.0,12.0,8.0,1.0,6.0,7.0,NaN,NaN,NaN,NaN
2017-06,3037.0,15.0,12.0,13.0,9.0,12.0,11.0,7.0,4.0,6.0,9.0,11.0,5.0,5.0,7.0,NaN,NaN,NaN,NaN,NaN


In [18]:
retention = cohort_matrix.div(cohort_matrix.iloc[:, 0], axis = 0) * 100
retention.round(2)

month_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2016-09,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,100.0,NaN,NaN,NaN,NaN,NaN,0.38,NaN,NaN,0.38,NaN,0.38,NaN,0.38,NaN,0.38,NaN,0.38,0.76,0.76
2016-12,100.0,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01,100.0,0.28,0.28,0.14,0.42,0.14,0.42,0.14,0.14,NaN,0.42,0.14,0.70,0.42,0.14,0.14,0.28,0.42,0.14,NaN
2017-02,100.0,0.18,0.31,0.12,0.43,0.12,0.25,0.18,0.12,0.18,0.12,0.31,0.12,0.18,0.12,0.06,0.06,0.18,NaN,NaN
2017-03,100.0,0.44,0.36,0.40,0.36,0.16,0.16,0.32,0.32,0.08,0.36,0.12,0.20,0.12,0.16,0.24,0.08,0.12,NaN,NaN
2017-04,100.0,0.62,0.22,0.18,0.27,0.27,0.35,0.31,0.31,0.18,0.27,0.09,0.04,0.04,0.09,0.09,0.13,NaN,NaN,NaN
2017-05,100.0,0.46,0.46,0.29,0.29,0.32,0.41,0.14,0.26,0.26,0.26,0.35,0.23,0.03,0.17,0.20,NaN,NaN,NaN,NaN
2017-06,100.0,0.49,0.40,0.43,0.30,0.40,0.36,0.23,0.13,0.20,0.30,0.36,0.16,0.16,0.23,NaN,NaN,NaN,NaN,NaN
